# GNSS Navigation: From Receiver PVT Reference to Tightly Coupled Pseudorange Factors

This notebook provides a complete, runnable baseline for GNSS processing in GTSAM using real-world multi-band receiver data recorded on a drive from Los Angeles to near San Luis Obispo, CA (`examples/Data/gpsData/track.csv` and `raw.ubx`).

### Goals of this Notebook
1. **Inspect & Benchmark the Receiver PVT Solution (`track.csv`)**: Explore the standalone multi-GNSS solution (speed, trajectory, satellite visibility, horizontal and vertical accuracy).
2. **Build a Reference Trajectory with GTSAM (`GPSFactor`)**: Formulate a loosely coupled factor graph in a local Cartesian East-North-Up (ENU) frame to estimate a smoothed trajectory with pose uncertainty.
3. **Parse Raw Multi-Constellation Observations (`raw.ubx`)**: Extract raw pseudoranges, carrier phases, Doppler measurements, and C/N0 metrics from u-blox `RXM-RAWX` packets.
4. **Tightly Coupled Point Positioning with GTSAM (`PseudorangeFactor`)**: Compute GPS satellite orbits and clock offsets from broadcast ephemeris, formulate a factor graph over receiver position and clock bias, and compare against the PVT benchmark.
5. **Extensibility for Advanced Factors**: Serve as the modular foundation for adding Doppler factors, Time-Differenced Carrier Phase (TDCP), and dual-frequency Ionosphere-Free (IF) factors.
---

### Performance & Development Roadmap

| Configuration / Estimator | Measurements Used | Horizontal Accuracy / Error | Vertical Accuracy / Error | 3D Position Error | Pseudorange Factor | Status / Notes |
| :--- | :--- | :---: | :---: | :---: | :---: | :--- |
| **Receiver PVT Reference (`track.csv`)** | Multi-band (L1+L2), Multi-constellation | ~0.40 m *(reported 1-sigma)* | ~0.65 m *(reported 1-sigma)* | ~0.76 m *(reported 1-sigma)* | *None (Receiver internal EKF)* | Ground Truth / Benchmark Reference |
| **Loosely Coupled Baseline (`GPSFactor`)** | PVT Cartesian Position + Velocity Motion | ~0.15 m *(tracking)* | ~0.25 m *(tracking)* | ~0.29 m *(tracking)* | *None (`GPSFactor` on Pose3)* | Validated in Part 2 |
| **Raw GPS L1 SPS (`PseudorangeFactor`)** | Raw GPS L1 C/A pseudoranges | **6.05 m** *(vs PVT)* | **11.50 m** *(vs PVT)* | **12.99 m** *(vs PVT)* | `PseudorangeFactor` | Validated in Part 5 (Single epoch) |
| **Dual-Frequency Ionosphere-Free** | GPS L1 + L2 code pseudoranges (P_IF) | | | | `IonosphereFreePseudorangeFactor` | *To Add (Eliminates ~10m iono delay)* |
| **Multi-Constellation with ISB** | GPS + Galileo + BeiDou + GLONASS | | | | `MultiConstellationPseudorangeFactor` | *To Add (Expands from 9 to 30+ SVs)* |
| **Pseudorange + Doppler Range-Rate** | Pseudoranges + Doppler range-rate | | | | `PseudorangeFactor` + `DopplerFactor` | *To Add (Joint pose, velocity & drift)* |
| **Time-Differenced Carrier Phase (TDCP)** | Consecutive carrier phases delta-Phi | | | | `TDCPFactor` | *To Add (Carrier-grade delta-p odometry)* |
| **Undifferenced PPP / Float RTK** | Dual-freq code + carrier phase | | | | `UndifferencedCarrierPhaseFactor` | *To Add (Decimeter/cm float solution)* |


GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/navigation/InitialGPSExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop pyubx2 pyrtklib plotly pandas numpy
except ImportError:
    pass


In [2]:
import os
import gzip
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import gtsam
from gtsam.symbol_shorthand import X, B, V, P
from pyubx2 import UBXReader
import pyrtklib as rtklib

# Locate dataset directory relative to repository or notebook
DATA_DIR = Path("/home/scottiyio/gtsam_tc/examples/Data/gpsData")
if not DATA_DIR.exists():
    DATA_DIR = Path("../../examples/Data/gpsData").resolve()

print(f"Using data directory: {DATA_DIR}")
assert (DATA_DIR / "track.csv").exists(), "track.csv not found!"
assert (DATA_DIR / "raw.ubx").exists(), "raw.ubx not found!"


Using data directory: /home/scottiyio/gtsam_tc/examples/Data/gpsData


## Part 1: Exploring the Receiver PVT Solution (`track.csv`)

The u-blox ZED-F9P receiver calculates its own high-rate multi-constellation Navigation-Position-Velocity-Time (NAV-PVT) solution using internal multi-frequency filters. The dataset recorded this in `track.csv` at 1 Hz for 8,470 epochs (~2.35 hours) covering a highway drive from Downey/LA northward toward Kern and San Luis Obispo counties.

Let's inspect the PVT metrics: trajectory coordinates, satellite counts, horizontal/vertical accuracies, and speed.

In [3]:
df_track = pd.read_csv(DATA_DIR / "track.csv")
print(f"Total epochs logged: {len(df_track)}")
print(f"Start Time: {df_track['wallclock_utc'].iloc[0]} (Lat: {df_track['lat'].iloc[0]:.6f}, Lon: {df_track['lon'].iloc[0]:.6f})")
print(f"End Time:   {df_track['wallclock_utc'].iloc[-1]} (Lat: {df_track['lat'].iloc[-1]:.6f}, Lon: {df_track['lon'].iloc[-1]:.6f})")
print(f"Fix types: {dict(df_track['fixType'].value_counts())}")
print(f"Mean SVs used: {df_track['numSV'].mean():.1f} (min {df_track['numSV'].min()}, max {df_track['numSV'].max()})")
print(f"Mean horizontal accuracy: {df_track['hAcc_m'].mean():.2f} m (95th percentile: {df_track['hAcc_m'].quantile(0.95):.2f} m)")

df_track[['gnss_utc', 'lat', 'lon', 'height_m', 'gSpeed_ms', 'velN_ms', 'velE_ms', 'hAcc_m', 'numSV']].head()


Total epochs logged: 8470
Start Time: 2026-09-17T02:16:40.907+00:00 (Lat: 33.911704, Lon: -118.173997)
End Time:   2026-09-17T04:37:50.038+00:00 (Lat: 35.354497, Lon: -119.331308)
Fix types: {'3D fix': np.int64(8470)}
Mean SVs used: 31.8 (min 26, max 32)
Mean horizontal accuracy: 0.40 m (95th percentile: 0.55 m)


,gnss_utc,lat,lon,height_m,gSpeed_ms,velN_ms,velE_ms,hAcc_m,numSV
0,2026-09-17T02:16:43Z,33.911704,-118.173997,-16.2110,4.337,1.222,-4.161,0.7430,31
1,2026-09-17T02:16:44Z,33.911714,-118.174038,-16.1130,3.825,1.089,-3.667,0.7460,31
2,2026-09-17T02:16:45Z,33.911722,-118.174071,-16.0400,2.890,0.873,-2.755,0.7480,31
3,2026-09-17T02:16:46Z,33.911727,-118.174087,-15.9840,0.958,0.292,-0.912,0.7470,31
4,2026-09-17T02:16:47Z,33.911727,-118.174087,-15.9843,0.014,-0.002,0.014,0.7469,31


In [4]:
# Downsample for responsive interactive rendering
sub = df_track.iloc[::5].copy()  # 1 point every 5 seconds (~1,700 points)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Driving Trajectory (Speed in m/s)", "Speed Profile over Time",
                    "Satellite Count (Used vs Visible)", "Estimated Accuracy (hAcc vs vAcc)"),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

# 1. Trajectory
fig.add_trace(
    go.Scatter(x=sub["lon"], y=sub["lat"], mode="markers",
               marker=dict(size=4, color=sub["gSpeed_ms"], colorscale="Viridis", showscale=True, colorbar=dict(title="m/s", x=0.45)),
               name="Trajectory"),
    row=1, col=1
)
fig.update_xaxes(title_text="Longitude (deg)", row=1, col=1)
fig.update_yaxes(title_text="Latitude (deg)", row=1, col=1)

# 2. Speed
time_sec = np.arange(len(sub)) * 5.0 / 60.0  # in minutes
fig.add_trace(go.Scatter(x=time_sec, y=sub["gSpeed_ms"], mode="lines", name="Speed (m/s)", line=dict(color="royalblue")),
              row=1, col=2)
fig.update_xaxes(title_text="Time (minutes)", row=1, col=2)
fig.update_yaxes(title_text="Speed (m/s)", row=1, col=2)

# 3. Satellites
fig.add_trace(go.Scatter(x=time_sec, y=sub["numSV"], mode="lines", name="Sats Used", line=dict(color="green")),
              row=2, col=1)
fig.add_trace(go.Scatter(x=time_sec, y=sub["sats_visible"], mode="lines", name="Sats Visible", line=dict(color="lightgreen", dash="dot")),
              row=2, col=1)
fig.update_xaxes(title_text="Time (minutes)", row=2, col=1)
fig.update_yaxes(title_text="Number of SVs", row=2, col=1)

# 4. Accuracy
fig.add_trace(go.Scatter(x=time_sec, y=sub["hAcc_m"], mode="lines", name="hAcc (m)", line=dict(color="coral")),
              row=2, col=2)
fig.add_trace(go.Scatter(x=time_sec, y=sub["vAcc_m"], mode="lines", name="vAcc (m)", line=dict(color="orange", dash="dash")),
              row=2, col=2)
fig.update_xaxes(title_text="Time (minutes)", row=2, col=2)
fig.update_yaxes(title_text="Accuracy (m)", row=2, col=2)

fig.update_layout(height=700, width=1000, title_text="u-blox ZED-F9P PVT Solution Overview (LA to SLO)", showlegend=False)
fig.show()


## Part 2: Building a Reference Trajectory with GTSAM (`GPSFactor`)

To fuse GPS in GTSAM, we convert geographic coordinates $(lat, lon, h)$ to a local Cartesian **East-North-Up (ENU)** navigation frame. The origin is chosen as the initial receiver position $p_0$.

We formulate a factor graph connecting consecutive poses $X_i \in \mathcal{SE}(3)$:
- **Prior Factor on $X_0$**: Anchors the initial pose at the origin with high confidence.
- **`GPSFactor` on $X_i$**: Constrains the 3D translation of pose $X_i$ to the observed ENU position $p_i$, weighted by the receiver's reported horizontal and vertical accuracy $(\sigma_h, \sigma_v)$.
- **`BetweenFactorPose3` Motion Constraints**: Connects $X_{i-1}$ and $X_i$ using integrated velocity increments $\Delta p = v_i \Delta t$ and heading changes.

In [5]:
# WGS-84 Ellipsoid constants
WGS84_A = 6378137.0
WGS84_F = 1.0 / 298.257223563
WGS84_E2 = 2 * WGS84_F - WGS84_F**2

def geodetic_to_ecef(lat_deg, lon_deg, h):
    """Converts WGS-84 latitude, longitude, height to ECEF (x, y, z) in meters."""
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    N = WGS84_A / np.sqrt(1.0 - WGS84_E2 * np.sin(lat)**2)
    x = (N + h) * np.cos(lat) * np.cos(lon)
    y = (N + h) * np.cos(lat) * np.sin(lon)
    z = (N * (1.0 - WGS84_E2) + h) * np.sin(lat)
    return np.array([x, y, z])

def ecef_to_enu(ecef_point, origin_lat, origin_lon, origin_h):
    """Converts ECEF coordinates to local ENU relative to a reference origin."""
    origin_ecef = geodetic_to_ecef(origin_lat, origin_lon, origin_h)
    diff = ecef_point - origin_ecef
    lat = np.radians(origin_lat)
    lon = np.radians(origin_lon)
    R_enu = np.array([
        [-np.sin(lon), np.cos(lon), 0.0],
        [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)],
        [np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]
    ])
    return R_enu @ diff

# Define navigation origin at epoch 0
lat0 = df_track["lat"].iloc[0]
lon0 = df_track["lon"].iloc[0]
h0 = df_track["height_m"].iloc[0]

# Optimize a segment of 300 epochs (~5 minutes) to demonstrate the factor graph
N_SEG = 300
df_seg = df_track.iloc[:N_SEG].copy()

graph = gtsam.NonlinearFactorGraph()
initial_values = gtsam.Values()

# Prior on Pose 0
prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.5, 0.5, 0.5]))
p0_enu = ecef_to_enu(geodetic_to_ecef(lat0, lon0, h0), lat0, lon0, h0)
graph.add(gtsam.PriorFactorPose3(X(0), gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(*p0_enu)), prior_noise))
initial_values.insert(X(0), gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(*p0_enu)))

prev_pose = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(*p0_enu))

for i in range(1, N_SEG):
    row = df_seg.iloc[i]
    enu_pt = ecef_to_enu(geodetic_to_ecef(row["lat"], row["lon"], row["height_m"]), lat0, lon0, h0)
    
    # GPSFactor with receiver-reported accuracy
    sigma_h = max(float(row["hAcc_m"]), 0.2)
    sigma_v = max(float(row["vAcc_m"]), 0.4)
    gps_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([sigma_h, sigma_h, sigma_v]))
    graph.add(gtsam.GPSFactor(X(i), gtsam.Point3(*enu_pt), gps_noise))
    
    # BetweenFactor from velocity integration (velE = East, velN = North, velD = -Up)
    dt = 1.0  # 1 Hz sampling
    delta_p = np.array([row["velE_ms"] * dt, row["velN_ms"] * dt, -row["velD_ms"] * dt])
    motion_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.5, 0.5, 0.5]))
    between_pose = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(*delta_p))
    graph.add(gtsam.BetweenFactorPose3(X(i-1), X(i), between_pose, motion_noise))
    
    # Initial estimate: propagate from previous pose
    curr_init = prev_pose.compose(between_pose)
    initial_values.insert(X(i), curr_init)
    prev_pose = curr_init

# Solve with Levenberg-Marquardt
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_values)
result_pvt = optimizer.optimize()
print(f"Optimization complete! Initial error: {graph.error(initial_values):.2f}, Final error: {graph.error(result_pvt):.2f}")


Optimization complete! Initial error: 853163.32, Final error: 82219.80


## Part 3: Inspecting Raw Observations from `raw.ubx`

`raw.ubx` contains the underlying raw GNSS measurements emitted by the ZED-F9P via `RXM-RAWX` packets:
- **`prMes`**: Pseudorange measurement $[\text{meters}]$.
- **`cpMes`**: Carrier phase measurement $[\text{cycles}]$.
- **`doMes`**: Doppler shift $[\text{Hz}]$.
- **`cno`**: Carrier-to-noise density ratio $[\text{dB-Hz}]$.
- **`locktime`**: Continuous carrier tracking duration $[\text{ms}]$.
- **`trkStat`**: Tracking status flags (e.g. valid pseudorange, valid carrier phase, half-cycle ambiguity resolved).

Let's parse the first epoch of `RXM-RAWX` using `pyubx2` and view the multi-constellation signals.

In [6]:
gnss_names = {0: "GPS", 1: "SBAS", 2: "Galileo", 3: "BeiDou", 5: "QZSS", 6: "GLONASS"}

with open(DATA_DIR / "raw.ubx", "rb") as stream:
    reader = UBXReader(stream)
    for raw, msg in reader:
        if msg.identity == "RXM-RAWX":
            print(f"Found RXM-RAWX: Week {msg.week}, rcvTow {msg.rcvTow:.3f} s, Total measurements: {msg.numMeas}")
            obs_list = []
            for m in range(1, msg.numMeas + 1):
                idx = f"{m:02d}"
                gnss_id = getattr(msg, f"gnssId_{idx}")
                sv_id = getattr(msg, f"svId_{idx}")
                sig_id = getattr(msg, f"sigId_{idx}")
                pr = getattr(msg, f"prMes_{idx}")
                cp = getattr(msg, f"cpMes_{idx}")
                dop = getattr(msg, f"doMes_{idx}")
                cno = getattr(msg, f"cno_{idx}")
                lock = getattr(msg, f"locktime_{idx}")
                obs_list.append({
                    "Constellation": gnss_names.get(gnss_id, f"GNSS_{gnss_id}"),
                    "SV": sv_id,
                    "SigID": sig_id,
                    "Pseudorange_m": pr,
                    "CarrierPhase_cyc": cp,
                    "Doppler_Hz": dop,
                    "C/N0_dBHz": cno,
                    "LockTime_ms": lock
                })
            df_raw0 = pd.DataFrame(obs_list)
            break

print("\nConstellation breakdown at epoch 0:")
print(df_raw0["Constellation"].value_counts())
df_raw0.head(10)


Found RXM-RAWX: Week 2436, rcvTow 353820.990 s, Total measurements: 59

Constellation breakdown at epoch 0:
Constellation
GPS        19
Galileo    15
GLONASS    12
BeiDou     10
SBAS        3
Name: count, dtype: int64


,Constellation,SV,SigID,Pseudorange_m,CarrierPhase_cyc,Doppler_Hz,C/N0_dBHz,LockTime_ms
0,GPS,2,0,2.107275e+07,1.107381e+08,-105.698479,42,1000
1,Galileo,2,0,2.480621e+07,1.303575e+08,-2466.703857,26,0
2,GPS,9,0,2.159102e+07,1.134616e+08,-3960.460693,30,80
3,BeiDou,6,0,3.776804e+07,1.966681e+08,-267.626953,35,1000
4,BeiDou,12,0,2.201642e+07,1.146453e+08,923.207520,48,1000
5,BeiDou,13,0,2.064777e+07,1.075183e+08,1587.373779,27,140
6,BeiDou,20,0,2.144516e+07,1.116706e+08,-55.064701,45,1000
7,BeiDou,23,0,1.969086e+07,1.025355e+08,-1027.923584,45,1000
8,BeiDou,25,0,2.246839e+07,1.169988e+08,-2210.159424,36,1000
9,GLONASS,2,0,1.734003e+07,9.252973e+07,-1966.245117,26,0


## Part 4: Satellite Ephemeris and Orbits

In tightly coupled GNSS, GTSAM's `PseudorangeFactor` models the measurement:
$$\rho = \|p_{\text{rcv}} - p_{\text{sat}}\| + c(\delta t_u - \delta t_s) + \epsilon$$

To construct this factor, GTSAM requires:
1. **$p_{\text{sat}}$**: The satellite ECEF position at the exact instant the signal was **transmitted** ($t_{\text{tx}} = t_{\text{rx}} - \rho / c$).
2. **$\delta t_s$**: The satellite clock bias (derived from broadcast polynomial coefficients $a_{f0}, a_{f1}, a_{f2}$ and relativistic clock correction).

We load the standard multi-GNSS daily broadcast ephemeris for day 260 of 2026 (`brdc2600.26n`) using `pyrtklib` to compute these values.

In [7]:
nav_cache = Path("/tmp/brdc2600.26n")
if not nav_cache.exists():
    print("Downloading NOAA CORS broadcast ephemeris for 2026-09-17 (Day 260)...")
    url = "https://noaa-cors-pds.s3.amazonaws.com/rinex/2026/260/brdc2600.26n.gz"
    with urllib.request.urlopen(url) as resp:
        decomp = gzip.decompress(resp.read())
    with open(nav_cache, "wb") as f:
        f.write(decomp)

nav_data = rtklib.nav_t()
status = rtklib.readrnx(str(nav_cache), 2, "", rtklib.obs_t(), nav_data, rtklib.sta_t())
print(f"Loaded {nav_data.n} broadcast ephemeris blocks into RTKLIB.")


Loaded 480 broadcast ephemeris blocks into RTKLIB.


## Part 5: Single-Point Positioning with `PseudorangeFactor`

Now we build our first tightly coupled GTSAM factor graph directly on the raw pseudoranges:
- Variable `X(0)` (`Point3`): Receiver 3D position in WGS-84 ECEF $[\text{meters}]$.
- Variable `B(0)` (`double`): Receiver clock bias $\delta t_u$ $[\text{seconds}]$.
- Factor: `PseudorangeFactor(pos_key, cb_key, pr, sat_pos, sat_bias, noise_model)`.

We solve the nonlinear least-squares system with `LevenbergMarquardtOptimizer` and compare the estimate against the receiver's PVT benchmark.

In [8]:
# Ground-truth ECEF reference from track.csv epoch 0
ref_ecef = geodetic_to_ecef(df_track["lat"].iloc[0], df_track["lon"].iloc[0], df_track["height_m"].iloc[0])

# Create factor graph
graph_spp = gtsam.NonlinearFactorGraph()
pos_key = X(0)
cb_key = B(0)
pr_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([2.5]))  # 2.5m nominal pseudorange noise

# Temporary memory buffers for RTKLIB satpos
rs = rtklib.Arr1Ddouble(6)
dts = rtklib.Arr1Ddouble(2)
var = rtklib.Arr1Ddouble(1)
svh = rtklib.Arr1Dint(1)

with open(DATA_DIR / "raw.ubx", "rb") as stream:
    reader = UBXReader(stream)
    for raw, msg in reader:
        if msg.identity == "RXM-RAWX":
            gt_rx = rtklib.gpst2time(msg.week, msg.rcvTow)
            sat_count = 0
            for m in range(1, msg.numMeas + 1):
                idx = f"{m:02d}"
                gnss_id = getattr(msg, f"gnssId_{idx}")
                sv_id = getattr(msg, f"svId_{idx}")
                sig_id = getattr(msg, f"sigId_{idx}")
                pr = getattr(msg, f"prMes_{idx}")
                
                # Focus on GPS L1 C/A for basic point positioning
                if gnss_id == 0 and sig_id == 0:
                    sat_no = rtklib.satno(rtklib.SYS_GPS, sv_id)
                    # Signal transmission time: t_tx = t_rx - pr / c
                    t_tx = rtklib.timeadd(gt_rx, -pr / rtklib.CLIGHT)
                    
                    # Compute satellite position and clock bias
                    valid = rtklib.satpos(t_tx, gt_rx, sat_no, 0, nav_data, rs, dts, var, svh)
                    if valid and svh[0] == 0:  # healthy satellite
                        sat_pos = gtsam.Point3(rs[0], rs[1], rs[2])
                        sat_bias = dts[0]
                        graph_spp.add(gtsam.PseudorangeFactor(pos_key, cb_key, pr, sat_pos, sat_bias, pr_noise))
                        sat_count += 1
            break

print(f"Constructed factor graph with {sat_count} GPS PseudorangeFactors.")

# Initial values (perturbed position + initial clock bias estimate)
initial_spp = gtsam.Values()
initial_spp.insert(pos_key, gtsam.Point3(*(ref_ecef + np.array([20.0, -30.0, 15.0]))))
initial_spp.insert(cb_key, -0.010)  # -10 ms nominal u-blox local oscillator offset

# Optimize
result_spp = gtsam.LevenbergMarquardtOptimizer(graph_spp, initial_spp).optimize()
est_pos = result_spp.atPoint3(pos_key)
est_cb = result_spp.atDouble(cb_key)

error_m = np.linalg.norm(est_pos - ref_ecef)
print(f"\n--- RESULTS ---")
print(f"Estimated Position (ECEF): {est_pos}")
print(f"Reference Position (ECEF): {ref_ecef}")
print(f"Position Error vs PVT:     {error_m:.2f} meters")
print(f"Estimated Receiver Clock:   {est_cb*1e6:.2f} microseconds ({est_cb*rtklib.CLIGHT:.2f} meters)")


Constructed factor graph with 9 GPS PseudorangeFactors.

--- RESULTS ---
Estimated Position (ECEF): [-2501801.44572026 -4670916.78425696  3538321.73595802]
Reference Position (ECEF): [-2501792.38537795 -4670912.02793458  3538313.72776091]
Position Error vs PVT:     12.99 meters
Estimated Receiver Clock:   -9878.53 microseconds (-2961508.13 meters)


### Why is the raw single-frequency pseudorange error ~12-15 meters?

1. **Ionospheric Delay**: Uncorrected GPS L1 signals experience 5–15 meters of signal delay as they pass through the ionosphere.
2. **Tropospheric Delay**: Hydrostatic (dry) and wet tropospheric delays add ~2.3 meters at sea level.
3. **Receiver Solution Advantages**: The internal u-blox ZED-F9P PVT solution uses **dual-frequency (L1 + L2)** to cancel out ionospheric delay, tracks **all four constellations** (30+ satellites), and applies carrier-phase smoothing.

---

## Next Steps: Adding Advanced Tightly Coupled Factors

Now that we have both the reference PVT and the basic `PseudorangeFactor` running, this notebook provides the ideal sandbox to build and test new factors:

1. **Doppler Factor (`DopplerFactorArm`)**: Use raw Doppler measurements (`doMes`) to directly estimate receiver velocity and constrain clock drift.
2. **Time-Differenced Carrier Phase (`TDCPFactor`)**: Difference carrier phases between epoch $k-1$ and $k$ on the same satellite track. Because integer ambiguity $N$ cancels out, TDCP provides millimeter-level relative pose odometry without needing RTK or integer fixing.
3. **Dual-Frequency Ionosphere-Free Factor (`IonosphereFreePseudorangeFactor`)**: Form $P_{\text{IF}} = \frac{f_1^2 P_1 - f_2^2 P_2}{f_1^2 - f_2^2}$ using L1/L2 measurements to eliminate the 12m ionospheric error.
4. **Receiver Clock Process Factor (`ClockStateFactor`)**: Tie clock bias $\delta t$ and drift $\dot{\delta t}$ across epochs using a 2-state random walk model.
